In [32]:
# Instalación de Streamlit (solo si hace falta)
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
# Instalación de Plotly, pandas, NumPy, SciPy, scikit-learn, matplotlib y openpyxl
%pip install plotly pandas numpy scipy scikit-learn matplotlib openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
%%writefile CLV_Historico_ES.py
# ============================================================
# DASHBOARD CLV: HISTÓRICO DE ENTRADAS Y SALIDAS
# Adaptación de la plantilla "Despliegue.ipynb" a la hoja
# "Histórico_Entradas_SAlidas" del archivo Datos Complementarios CLV.
# ============================================================

# 1. IMPORTACIÓN DE LIBRERÍAS
import streamlit as st
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, confusion_matrix, accuracy_score, precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import stats

# 2. CONFIGURACIÓN GENERAL DE LA APLICACIÓN
st.set_page_config(page_title="CLV | Histórico de entradas y salidas", page_icon="📦", layout="wide")

# 3. CARGA Y PREPARACIÓN DE DATOS
@st.cache_data
def load_data():

    archivo = "Datos_Complementarios_CLV.xlsx"
    hoja = "Histórico_Entradas_SAlidas"
    df = pd.read_excel(archivo, sheet_name=hoja)

    # Normalizamos espacios en los nombres de columnas por "seguridad".
    df.columns = df.columns.astype(str).str.strip()

    # Convertimos campos numéricos a números; los valores no convertibles
    # se convierten en NaN y se tratarán según el análisis seleccionado.
    for columna in ["ANO", "MES", "SALIDAS"]:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")

    # Variables para extracción de características. ALMACEN solo tiene
    # una categoría en esta hoja, por lo que no es útil para comparar grupos.
    lista_categoricas = ["MARCA", "MES", "ANO"]
    lista_numericas = ["ANO", "MES", "SALIDAS"]
    return df, lista_categoricas, lista_numericas

df, Lista_cat, Lista_num = load_data()

# 4. ENCABEZADO Y NAVEGACIÓN LATERAL
st.sidebar.title("📦 HISTÓRICO CLV")
View = st.sidebar.selectbox(
    label="Tipo de Análisis",
    options=[
        "Extracción de Características",
        "Regresión Lineal",
        "Regresión No Lineal",
        "Regresión Logística",
        "ANOVA"
    ]
)

# Indicadores generales visibles para todas las vistas
st.title("Análisis del histórico de salidas", anchor=False)
m1, m2, m3 = st.columns(3)
m1.metric("Registros", f"{len(df):,}")
m2.metric("Unidades de salida", f"{df['SALIDAS'].sum():,.0f}")
m3.metric("Marcas", f"{df['MARCA'].nunique():,}")

#Mini divisor visual
st.divider()

# ------------------------------------------------------------
# VISTA 1. EXTRACCIÓN DE CARACTERÍSTICAS (ANÁLISIS UNIVARIADO)
# ------------------------------------------------------------

if View == "Extracción de Características":
    # La variable elegida puede ser marca, mes o año. Se cuentan sus frecuencias.
    Variable_Cat = st.sidebar.selectbox("Variable", options=Lista_cat, index=0)
    Tabla_frecuencias = (
        df[Variable_Cat].dropna().value_counts().rename_axis("categorias")
        .reset_index(name="frecuencia")
    )
    # Orden cronológico para MES y ANO; para MARCA se conserva el orden por frecuencia.
    if Variable_Cat in ["MES", "ANO"]:
        Tabla_frecuencias = Tabla_frecuencias.sort_values("categorias")

    st.header("Extracción de Características", anchor=False)
    st.write(f"Distribución de frecuencias de **{Variable_Cat}**.")

    # Se conserva la distribución en cuatro gráficos de la plantilla.
    Contenedor_A, Contenedor_B = st.columns(2)
    with Contenedor_A:
        st.write("Gráfico de barras")
        figure1 = px.bar(Tabla_frecuencias, x="categorias", y="frecuencia",
                         title=f"Frecuencia por {Variable_Cat}")
        figure1.update_xaxes(automargin=True)
        figure1.update_yaxes(automargin=True)
        figure1.update_layout(height=350)
        st.plotly_chart(figure1, use_container_width=True)

    with Contenedor_B:
        st.write("Gráfico de pastel")
        figure2 = px.pie(Tabla_frecuencias, names="categorias", values="frecuencia",
                         title=f"Proporción por {Variable_Cat}")
        figure2.update_layout(height=350)
        st.plotly_chart(figure2, use_container_width=True)

    Contenedor_C, Contenedor_D = st.columns(2)
    with Contenedor_C:
        st.write("Gráfico de anillo o dona")
        figure3 = px.pie(Tabla_frecuencias, names="categorias", values="frecuencia",
                         hole=0.4, title=f"Proporción por {Variable_Cat}")
        figure3.update_layout(height=350)
        st.plotly_chart(figure3, use_container_width=True)

    with Contenedor_D:
        st.write("Gráfico de área")
        figure4 = px.area(Tabla_frecuencias, x="categorias", y="frecuencia",
                          title=f"Frecuencia por {Variable_Cat}")
        figure4.update_layout(height=350)
        st.plotly_chart(figure4, use_container_width=True)

    st.dataframe(Tabla_frecuencias, use_container_width=True)

# ------------------------------------------------------------
# VISTA 2. REGRESIÓN LINEAL SIMPLE Y MÚLTIPLE
# ------------------------------------------------------------

elif View == "Regresión Lineal":
    st.header("Regresión Lineal")
    st.caption("La variable SALIDAS representa las unidades registradas como salida. ANO y MES son variables temporales numéricas.")
    # Para evitar elegir la misma variable como X y como Y, X se filtra después.
    Variable_y = st.sidebar.selectbox("Variable objetivo (Y)", options=Lista_num, index=2)
    opciones_x = [c for c in Lista_num if c != Variable_y]
    Variable_x = st.sidebar.selectbox("Variable independiente del modelo simple (X)", options=opciones_x)
    Variables_x = st.sidebar.multiselect(
        "Variables independientes del modelo múltiple (X)",
        options=opciones_x, default=opciones_x
    )

    # Eliminar nulos únicamente en las columnas que participan en cada modelo.
    datos_simple = df[[Variable_x, Variable_y]].dropna()
    if len(datos_simple) >= 2 and datos_simple[Variable_x].nunique() > 1:
        Contenedor_A, Contenedor_B = st.columns(2)
        with Contenedor_A:
            st.subheader("Regresión lineal simple")
            modelo_simple = LinearRegression()
            modelo_simple.fit(datos_simple[[Variable_x]], datos_simple[Variable_y])
            pred_simple = modelo_simple.predict(datos_simple[[Variable_x]])
            r2_simple = r2_score(datos_simple[Variable_y], pred_simple)
            correlacion = datos_simple[Variable_x].corr(datos_simple[Variable_y])
            st.metric("R²", f"{r2_simple:.4f}")
            st.metric("Correlación de Pearson", f"{correlacion:.4f}" if pd.notna(correlacion) else "No disponible")
            st.write(f"**Ecuación:** Ŷ = {modelo_simple.intercept_:.3f} + ({modelo_simple.coef_[0]:.3f} × {Variable_x})")
            fig5 = px.scatter(datos_simple, x=Variable_x, y=Variable_y, title="Modelo lineal simple")
            orden = np.argsort(datos_simple[Variable_x].to_numpy())
            fig5.add_trace(go.Scatter(
                x=datos_simple[Variable_x].to_numpy()[orden],
                y=pred_simple[orden], mode="lines", name="Predicción"
            ))
            st.plotly_chart(fig5, use_container_width=True)
        with Contenedor_B:
            st.subheader("Regresión lineal múltiple")
            if not Variables_x:
                st.info("Selecciona al menos una variable independiente para ajustar el modelo múltiple.")
            else:
                datos_multiple = df[Variables_x + [Variable_y]].dropna()
                if len(datos_multiple) >= 2:
                    modelo_multiple = LinearRegression()
                    modelo_multiple.fit(datos_multiple[Variables_x], datos_multiple[Variable_y])
                    pred_multiple = modelo_multiple.predict(datos_multiple[Variables_x])
                    st.metric("R² del modelo múltiple", f"{r2_score(datos_multiple[Variable_y], pred_multiple):.4f}")
                    coeficientes = pd.DataFrame({"Variable": Variables_x, "Coeficiente": modelo_multiple.coef_})
                    st.dataframe(coeficientes, use_container_width=True)
                    st.write(f"Intercepto: {modelo_multiple.intercept_:.4f}")
                    st.caption("El R² se calcula sobre los mismos datos usados para ajustar el modelo; no es una evaluación predictiva en datos nuevos.")
                else:
                    st.warning("No hay suficientes registros completos para ajustar el modelo.")
    else:
        st.warning("No hay suficientes datos o variación en X para ajustar la regresión simple.")

# ------------------------------------------------------------
# VISTA 3. REGRESIÓN NO LINEAL
# ------------------------------------------------------------

elif View == "Regresión No Lineal":
    st.header("Regresión No Lineal")
    st.caption("La plantilla separaba los datos por sexo. Esta hoja no contiene esa variable, así que la comparación se adapta a las dos marcas con más registros.")
    # Se seleccionan marcas con más observaciones para tener grupos suficientemente grandes.
    marcas_top = df["MARCA"].value_counts().head(2).index.tolist()
    if len(marcas_top) < 2:
        st.warning("No hay dos marcas con registros suficientes para comparar.")
    else:
        marca_a = st.sidebar.selectbox("Grupo A (marca)", marcas_top, index=0)
        marca_b = st.sidebar.selectbox("Grupo B (marca)", marcas_top, index=1)
        Variable_y = st.sidebar.selectbox("Variable objetivo (Y)", options=["SALIDAS", "MES", "ANO"], index=0)
        Variable_x = st.sidebar.selectbox("Variable independiente (X)", options=[c for c in ["MES", "ANO"] if c != Variable_y], index=0)
        Modelo = st.sidebar.selectbox("Modelo no lineal", ["Función cuadrática", "Función exponencial"])

        def cuadratica(x, a, b, c):
            return a * x**2 + b * x + c

        def exponencial(x, a, b, c):
            return a * np.exp(-b * x) + c

        funcion = cuadratica if Modelo == "Función cuadrática" else exponencial
        Contenedor_A, Contenedor_B = st.columns(2)
        for contenedor, marca, titulo in [
            (Contenedor_A, marca_a, "Grupo A"),
            (Contenedor_B, marca_b, "Grupo B")
        ]:
            with contenedor:
                st.subheader(f"{titulo}: {marca}")
                datos = df.loc[df["MARCA"] == marca, [Variable_x, Variable_y]].dropna()
                # curve_fit necesita suficientes observaciones y variación en X.
                if len(datos) < 4 or datos[Variable_x].nunique() < 3:
                    st.warning("No hay suficientes datos distintos para ajustar este modelo.")
                    continue
                x = datos[Variable_x].astype(float).to_numpy()
                y = datos[Variable_y].astype(float).to_numpy()
                try:
                    parametros, _ = curve_fit(funcion, x, y, maxfev=20000)
                    y_pred = funcion(x, *parametros)
                    r2 = r2_score(y, y_pred)
                    st.metric("R²", f"{r2:.4f}")
                    # Ordenar X permite dibujar la curva de predicción correctamente.
                    orden = np.argsort(x)
                    fig = px.scatter(datos, x=Variable_x, y=Variable_y, title=f"{Modelo} — {marca}")
                    fig.add_trace(go.Scatter(x=x[orden], y=y_pred[orden], mode="lines", name="Predicción"))
                    st.plotly_chart(fig, use_container_width=True)
                except (RuntimeError, ValueError, FloatingPointError) as error:
                    st.warning(f"No se pudo ajustar el modelo con esta combinación: {error}")

# ------------------------------------------------------------
# VISTA 4. REGRESIÓN LOGÍSTICA
# ------------------------------------------------------------

elif View == "Regresión Logística":
    st.header("Regresión Logística")
    st.caption("La hoja no contiene una variable binaria como Supervivencia. Para conservar el ejercicio, se crea una etiqueta: salida alta (> mediana) frente a salida baja o igual a la mediana.")
    # Variable objetivo derivada de SALIDAS. La mediana evita una división arbitraria por valor fijo.
    mediana_salidas = df["SALIDAS"].median()
    datos_log = df[["ANO", "MES", "SALIDAS"]].dropna().copy()
    datos_log["Nivel_salida"] = (datos_log["SALIDAS"] > mediana_salidas).astype(int)
    Variables_x = st.sidebar.multiselect(
        "Variables independientes (X)", options=["ANO", "MES"],
        default=["ANO", "MES"]
    )
    if not Variables_x:
        st.warning("Selecciona al menos una variable independiente.")
    elif datos_log["Nivel_salida"].nunique() < 2:
        st.warning("La variable objetivo no tiene dos clases; no se puede ajustar una regresión logística.")
    else:
        X = datos_log[Variables_x]
        y = datos_log["Nivel_salida"]
        # Estratificar mantiene la proporción de clases en entrenamiento y prueba.
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.30, random_state=42, stratify=y
        )
        # El escalador se ajusta solo con entrenamiento para evitar fuga de información.
        escalar = StandardScaler()
        X_train_escalado = escalar.fit_transform(X_train)
        X_test_escalado = escalar.transform(X_test)
        algoritmo = LogisticRegression(max_iter=1000, class_weight="balanced")
        algoritmo.fit(X_train_escalado, y_train)
        y_pred = algoritmo.predict(X_test_escalado)

        Contenedor_A, Contenedor_B = st.columns(2)
        with Contenedor_A:
            st.subheader("Matriz de confusión")
            matriz = confusion_matrix(y_test, y_pred, labels=[0, 1])
            fig = go.Figure(data=go.Heatmap(
                z=matriz, x=["Predicción: baja", "Predicción: alta"],
                y=["Real: baja", "Real: alta"], hoverinfo="z",
                colorscale="Oranges", showscale=True
            ))
            for i in range(2):
                for j in range(2):
                    fig.add_annotation(x=j, y=i, text=str(matriz[i, j]), showarrow=False,
                                       font=dict(color="black"))
            fig.update_layout(title="Matriz de confusión", xaxis_title="Clase predicha",
                              yaxis_title="Clase real", height=420)
            st.plotly_chart(fig, use_container_width=True)
        with Contenedor_B:
            st.subheader("Métricas de clasificación")
            st.metric("Exactitud", f"{accuracy_score(y_test, y_pred):.3f}")
            st.metric("Precisión — salida baja", f"{precision_score(y_test, y_pred, pos_label=0, zero_division=0):.3f}")
            st.metric("Precisión — salida alta", f"{precision_score(y_test, y_pred, pos_label=1, zero_division=0):.3f}")
            st.write(f"**Umbral usado:** SALIDAS > {mediana_salidas:.0f} = salida alta.")
            st.caption("Las métricas evalúan la clasificación de registros, no predicen directamente el inventario disponible.")

# ------------------------------------------------------------
# VISTA 5. ANOVA DE UN FACTOR
# ------------------------------------------------------------

elif View == "ANOVA":
    st.header("ANOVA")
    # SALIDAS es cuantitativa; MARCA, MES y ANO se pueden comparar como grupos.
    Variable_y = st.sidebar.selectbox("Variable dependiente (Y)", options=["SALIDAS"], index=0)
    Variable_x = st.sidebar.selectbox("Variable independiente (grupo)", options=["MARCA", "MES", "ANO"], index=0)
    datos_anova = df[[Variable_x, Variable_y]].dropna()
    # Evitamos comparar grupos con muy pocos datos o una sola categoría.
    conteos = datos_anova.groupby(Variable_x)[Variable_y].size()
    grupos_validos = conteos[conteos >= 2].index
    datos_anova = datos_anova[datos_anova[Variable_x].isin(grupos_validos)]
    grupos = [g[Variable_y].to_numpy() for _, g in datos_anova.groupby(Variable_x)]
    Contenedor_A, Contenedor_B = st.columns(2)

    with Contenedor_A:
        st.subheader("Resultados del ANOVA")
        if len(grupos) >= 2:
            f_valor, p_valor = stats.f_oneway(*grupos)
            st.write(f"**F-valor:** {f_valor:.3f}")
            st.write(f"**p-valor:** {p_valor:.6g}")
            if p_valor < 0.05:
                st.success("Se detectan diferencias estadísticamente significativas entre al menos dos medias de grupo (α = 0.05).")
            else:
                st.warning("No se detectan diferencias estadísticamente significativas entre las medias (α = 0.05).")
            st.caption("ANOVA supone independencia, residuos aproximadamente normales y homogeneidad de varianzas. Un resultado significativo no indica por sí solo qué grupos difieren.")
        else:
            st.warning("Se necesitan al menos dos grupos válidos para ejecutar ANOVA.")

    with Contenedor_B:
        st.subheader("Gráfico de barras: media por grupo")
        resumen = datos_anova.groupby(Variable_x)[Variable_y].agg(["mean", "sem", "count"]).reset_index()
        if not resumen.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            x = resumen[Variable_x].astype(str)
            ax.bar(x, resumen["mean"], yerr=resumen["sem"].fillna(0), capsize=4,
                   color="skyblue", edgecolor="black")
            ax.set_ylabel(f"Media de {Variable_y}")
            ax.set_xlabel(Variable_x)
            ax.set_title("Media por grupo con error estándar")
            ax.tick_params(axis="x", rotation=45)
            fig.tight_layout()
            st.pyplot(fig)
    st.write("Resumen descriptivo por grupo")
    st.dataframe(resumen.rename(columns={"mean": "media", "sem": "error_estandar", "count": "registros"}), use_container_width=True)

# ------------------------------------------------------------
# NOTAS DE INTERPRETACIÓN (EXTRAS)
# ------------------------------------------------------------
#with st.expander("Notas sobre la adaptación de la plantilla"):
    #st.markdown("""
    #- **Extracción de características:** se conserva el diseño de cuatro gráficos; las categorías se obtienen de MARCA, MES o ANO.
    #- **Regresión lineal:** se utilizan las columnas numéricas ANO, MES y SALIDAS. El ajuste es exploratorio y no implica causalidad.
    #- **Regresión no lineal:** se sustituye la separación por sexo de Titanic por las dos marcas con más registros, ya que la hoja no contiene sexo.
    #- **Regresión logística:** se construye una variable binaria derivada de SALIDAS usando la mediana como umbral. Es una adaptación didáctica, no una etiqueta original de la base.
    #- **ANOVA:** se compara la media de SALIDAS entre marcas, meses o años. ALMACEN no se incluye porque en esta hoja solo aparece “Almacén general”.
    #- **Histórico:** la hoja contiene registros de salidas; no se debe interpretar SALIDAS como existencias actuales ni como entradas.
    #""")


Overwriting CLV_Historico_ES.py
